# Setup

In [15]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Pinecone as PineconeVectorStore
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain.schema import Document
from pinecone import Pinecone
import pandas as pd
import yaml
import os

In [16]:
with open('config.yaml', 'r') as config_file:
    config = yaml.safe_load(config_file)
os.environ['PINECONE_API_KEY'] = config['PINECONE_API_KEY']
os.environ['OPENAI_API_KEY'] = config['OPENAI_API_KEY']

# Busca Semântica

In [17]:
df = pd.read_csv('Base_Kaggle/Crop_recommendation.csv')

documents = []
for cultura, g in df.groupby('label'):
    texto = (
        f"Cultura: {cultura}. "
        f"Nitrogênio: {g['N'].mean():.0f}. Fósforo: {g['P'].mean():.0f}. Potássio: {g['K'].mean():.0f}. "
        f"Temperatura: {g['temperature'].mean():.1f}°C. Umidade: {g['humidity'].mean():.1f}%. "
        f"pH: {g['ph'].mean():.1f}. Chuva: {g['rainfall'].mean():.0f}mm."
    )
    documents.append(Document(page_content=texto))

In [18]:
pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
pc.Index('nlp').delete(delete_all=True)

{}

In [19]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-large', dimensions=2048)
index_name = 'nlp'  

In [20]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    length_function=len
)
chunks = text_splitter.create_documents([doc.page_content for doc in documents])

In [21]:
vector_store = PineconeVectorStore.from_documents(chunks, embeddings, index_name=index_name)

In [22]:
query_1 = 'Responda apenas com base no input fornecido. Qual cultura é recomendada para solo com alto nitrogênio?'
query_2 = 'Responda apenas com base no input fornecido. Quais culturas se adaptam a alta umidade?'
query_3 = 'Responda apenas com base no input fornecido. Qual cultura cresce bem em pH ácido?'
query_4 = 'Responda apenas com base no input fornecido. Qual cultura precisa de mais chuva?'

In [23]:
nlp = ChatOpenAI(model='gpt-3.5-turbo', temperature=0.2)

In [24]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k': 22})

In [25]:
chain = RetrievalQA.from_chain_type(llm=nlp, chain_type='stuff', retriever=retriever)

In [26]:
print(chain)

combine_documents_chain=StuffDocumentsChain(llm_chain=LLMChain(prompt=ChatPromptTemplate(input_variables=['context', 'question'], messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], template="Use the following pieces of context to answer the user's question. \nIf you don't know the answer, just say that you don't know, don't try to make up an answer.\n----------------\n{context}")), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], template='{question}'))]), llm=ChatOpenAI(client=<openai.resources.chat.completions.Completions object at 0x00000161680DB1D0>, async_client=<openai.resources.chat.completions.AsyncCompletions object at 0x00000161680E2690>, temperature=0.2, openai_api_key=SecretStr('**********'), openai_proxy='')), document_variable_name='context') retriever=VectorStoreRetriever(tags=['Pinecone', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.pinecone.Pinecone object at 0x0000016166F31290>, 

In [27]:
answer_1 = chain.invoke(query_1)
answer_2 = chain.invoke(query_2)
answer_3 = chain.invoke(query_3)
answer_4 = chain.invoke(query_4)

In [28]:
print('Pergunta: ',answer_1['query'])
print('Resultado: ',answer_1['result'],'\n')
#---
print('Pergunta: ',answer_2['query'])
print('Resultado: ',answer_2['result'],'\n')
#---
print('Pergunta: ',answer_3['query'])
print('Resultado: ',answer_3['result'],'\n')
#---
print('Pergunta: ',answer_4['query'])
print('Resultado: ',answer_4['result'])

Pergunta:  Responda apenas com base no input fornecido. Qual cultura é recomendada para solo com alto nitrogênio?
Resultado:  A cultura recomendada para solo com alto nitrogênio é a cultura de cotton, que requer 118 unidades de nitrogênio. 

Pergunta:  Responda apenas com base no input fornecido. Quais culturas se adaptam a alta umidade?
Resultado:  As culturas que se adaptam a alta umidade com base nas informações fornecidas são: muskmelon, watermelon, coconut, papaya. 

Pergunta:  Responda apenas com base no input fornecido. Qual cultura cresce bem em pH ácido?
Resultado:  A cultura que cresce bem em pH ácido é a cultura de pomegranate, com um pH de 6.4. 

Pergunta:  Responda apenas com base no input fornecido. Qual cultura precisa de mais chuva?
Resultado:  A cultura que precisa de mais chuva é o arroz, com 236mm de chuva necessária.
